# 초구 예측 — model_05 (2단계: 구종에 A 이번 타석 상황)

- 2단계 첫 모델: **베이스라인 구종 확률(B4, model_01 파라미터)을 출발점으로 두고, LightGBM이 상황에 따른 보정만 학습**
  - 출발점 = log P_B4 (init_score) -> 트리 출력을 더해 softmax -> 상황 정보가 쓸모없으면 B4 확률 그대로
  - 넣는 변수: A 이번 타석 상황 한 묶음 (0-2에서 고름)
    - 그 경기 앞선 투구 수, 같은 타자 몇 번째, 선발/불펜, 이닝, 아웃, 주자(1·2·3루), 타순, 점수 차 (모두 경기 기록, 그 타석 앞 정보)
- 학습·선택: 2017–18 학습 -> 2019로 LightGBM 파라미터·트리 수 선택 / 2020은 고른 값으로 2017–19 다시 학습해 평가
  - 2017은 지난 시즌 리그 비율이 없어 그 시즌 비율로 대신 (학습 데이터 출발점용, 2019·2020 확률은 그대로)
- 위치: model_03 그대로 (실제 구종 기준으로 고른 값이라 구종 모델과 무관) -> 실전 값 변화 = 순수한 구종 개선 효과
- 비교 대상: model_03
- **0장 = 모델 만들기 전 확인** (2단계 1·2번 항목, 이 모델 설계의 근거) -> 1장부터 모델

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys
if not os.path.exists("fp_common.py"):  # model/ 폴더에서 열면 한 단계 위(데이터·fp_common·results 있는 곳)로
    os.chdir("..")
sys.path.insert(0, os.getcwd())  # fp_common을 찾도록
import numpy as np
import pandas as pd
import fp_common as fc

MODEL = "model_05"

m, names = fc.load_data()
fp, y_type, y_zone, val, test = fc.first_pitches(m)
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]
print(f"초구 {len(fp):,}개 (검증 {val.sum():,} / 평가 {test.sum():,})")
import lightgbm as lgb
from scipy.special import log_softmax
pd.set_option("display.max_rows", 200)

초구 50,709개 (검증 13,458 / 평가 12,615)


## 0. 모델 전 확인

- 0-1: 지금 구종 확률(model_03)이 어떤 투수·상황에서 실제 구종을 놓치는지 -> 2단계 변수 후보
- 0-2: 후보 변수 전부가 지금 확률에 얼마나 더해 줄지 미리 재서 넣는 순서를 정함
- 0장 코드는 m·fp에 열을 더하므로 1장 전에 데이터를 다시 불러옴

### 0-1. 구종 포함률 낮은 투수와 원인

- 2단계 첫 항목: 누적 80% 보여 주기에서 실제 구종이 빠지는 공이 어떤 투수·상황에 몰리는지 확인 -> 2단계 변수 고를 때 참고
- 대상 모델: model_03 (구종 확률은 model_01과 같음, B4 반감 25·alpha 5.48·hand_k 5.48)
- 판단은 2019(검증) 기준, 2020은 같은 방향인지 확인만 (변수 선택에 2020을 쓰지 않기 위해)
- 용어
  - 포함: 보여 준 구종(누적 80%까지) 안에 실제 구종이 있음
  - 기대 포함률: 보여 준 구종 확률의 합 -> 확률이 맞으면 포함률 = 기대
  - gap = 포함 − 기대 (음수 = 모델 생각보다 자주 놓침)
  - llx = log loss − 기대 log loss(엔트로피) -> 양수 = 과신, 음수 = 과소 확신
  - rFAST = 직구 실제(0/1) − 직구 예측 확률 -> 양수 = 직구를 덜 예측
- 상황 변수는 3장 "평가 설계" 기준 (경기 기록은 그 타석 시점까지만, 모두 그 공 앞 정보)

In [ ]:
import os, sys
if not os.path.exists("fp_common.py"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import numpy as np
import pandas as pd
import fp_common as fc

DIAG_MODEL = "model_03" 

m, names = fc.load_data()
fp, y_type, y_zone, val, test = fc.first_pitches(m)
pp = fc.load_results([DIAG_MODEL])[DIAG_MODEL]
print(f"초구 {len(fp):,}개 (검증 {val.sum():,} / 평가 {test.sum():,})")

초구 50,709개 (검증 13,458 / 평가 12,615)


#### 0-1-1. 상황 변수

- 모든 공(구종 의심 공 포함) 기준으로 계산 -> 초구에 붙임
- `pc` 그 경기 그 투수의 앞선 투구 수 (0 = 그 경기 첫 타자) / `starter` 그 경기 첫 공이 1회 / `tto` 같은 타자 몇 번째 대결
- `rest` 등판 간격(일, 41명 밖 등판은 모름) / `sdiff` 투수 팀 기준 점수 차 / `n_prior` 전날까지 그 투수 초구 수

In [3]:
m = m.sort_values(fc.KEYS)
g = m.groupby(["gameID", "pitcher_code"])
m["pc"] = g.cumcount()  # 그 경기 앞선 투구 수
m["starter"] = g["inning"].transform("first").eq(1)  # 그 경기 첫 공이 1회 = 선발
m["tto"] = m.groupby(["gameID", "pitcher_code", "batter_code"])["pa_id"].transform(lambda s: s.ne(s.shift()).cumsum())  # 같은 타자 몇 번째 대결
gd = m.drop_duplicates(["pitcher_code", "gameID"])[["pitcher_code", "gameID", "game_date"]].sort_values("game_date")
gd["rest"] = gd.groupby("pitcher_code")["game_date"].diff().dt.days  # 등판 간격
m = m.merge(gd[["pitcher_code", "gameID", "rest"]], on=["pitcher_code", "gameID"], how="left")
m["sdiff"] = np.where(m["tb"].eq("T"), m["home_score"] - m["away_score"], m["away_score"] - m["home_score"])  # 투수 팀 기준 점수 차
CTX = m[fc.KEYS + ["pc", "starter", "tto", "rest", "sdiff"]]

_, N = fc.lookup(fc.prior_table(fp, ["pitcher_code"], fc.CLASSES[:1]), fp, ["pitcher_code"], fc.CLASSES[:1])
fp["n_prior"] = N[:, 0]  # 전날까지 그 투수 초구 수 (가중 없음)

def prep(year):  # 그 해 초구 + 예측 + 상황 + 공별 진단 값
    d = fc.align(pp, fp, year).merge(CTX, on=fc.KEYS, how="left", validate="one_to_one")
    P, y = d[fc.P_TYPE].to_numpy(), d["pitch_type"].map(fc.TYPE_IDX).to_numpy()
    i, s = np.arange(len(y)), fc.show_types(P)
    d["hit"] = s[i, y].astype(float)
    d["exp"] = (P * s).sum(1)  # 기대 포함률
    d["gap"] = d["hit"] - d["exp"]
    d["llx"] = -np.log(P[i, y]) + (P * np.log(np.clip(P, 1e-12, None))).sum(1)  # log loss − 엔트로피
    d["rFAST"] = (y == fc.TYPE_IDX["FAST"]) - P[:, fc.TYPE_IDX["FAST"]]
    d["month"] = d["game_date"].dt.month
    return d, P, y

D = {yr: prep(yr) for yr in (fc.VAL_YEAR, fc.TEST_YEAR)}
for yr, (d, _, _) in D.items():
    print(yr, f"포함률 {d.hit.mean():.3f} / 기대 {d.exp.mean():.3f}")

2019 포함률 0.891 / 기대 0.899
2020 포함률 0.901 / 기대 0.900


#### 0-1-2. 투수별 포함률

- 표준오차는 경기 단위 (같은 경기 공끼리 독립이 아님) -> z = gap / 표준오차
- 보정이 맞고 투수 차이가 잡음뿐이면 z² 합 ≈ 투수 수

In [4]:
def per_pitcher(d):
    gg = d.groupby(["pitcher_code", "gameID"])["gap"].agg(["sum", "count"])
    r = gg.groupby("pitcher_code").agg(s=("sum", "sum"), n=("count", "sum"), 경기=("sum", "size"))
    r["gap"] = r.s / r.n
    resid = gg["sum"] - gg["count"] * r["gap"].reindex(gg.index.get_level_values(0)).to_numpy()
    r["se"] = np.sqrt((resid ** 2).groupby(level=0).sum()) / r.n  # 경기 단위 표준오차
    r["z"] = r.gap / r.se
    a = d.groupby("pitcher_code").agg(포함률=("hit", "mean"), 기대=("exp", "mean"), llx=("llx", "mean"), rFAST=("rFAST", "mean"))
    out = a.join(r[["n", "경기", "gap", "se", "z"]])
    out.insert(0, "투수", names.reindex(out.index).to_numpy())
    return out.sort_values("gap")

PP = {yr: per_pitcher(D[yr][0]) for yr in D}
for yr, t in PP.items():
    print(f"{yr}: z² 합 {(t.z ** 2).sum():.1f} / 투수 {len(t)}명, |z| > 2: {', '.join(t.loc[t.z.abs() > 2, '투수'])}")
print("\n2019 (gap 낮은 순)")
display(PP[fc.VAL_YEAR].round(3))

2019: z² 합 120.4 / 투수 38명, |z| > 2: 김범수, 구창모, 양현종, 주권, 조상우, 함덕주, 김태훈, 박진형
2020: z² 합 61.2 / 투수 39명, |z| > 2: 양현종, 김재윤, 김대현

2019 (gap 낮은 순)


,투수,포함률,기대,llx,rFAST,n,경기,gap,se,z
pitcher_code,,,,,,,,,,
65769,김범수,0.830,0.886,0.066,-0.020,399,41,-0.056,0.025,-2.214
68043,김민,0.843,0.897,0.192,-0.034,657,27,-0.054,0.032,-1.711
64021,박세웅,0.858,0.907,0.072,-0.015,239,11,-0.049,0.027,-1.794
65933,구창모,0.834,0.879,0.102,0.001,403,22,-0.045,0.022,-2.017
69446,원태인,0.812,0.852,0.096,0.017,467,25,-0.041,0.025,-1.614
67266,박치국,0.886,0.925,0.120,-0.038,176,49,-0.038,0.026,-1.468
66145,김대현,0.865,0.891,0.103,-0.002,252,36,-0.026,0.032,-0.811
68341,안우진,0.849,0.874,0.053,-0.023,371,18,-0.025,0.020,-1.246
69045,알칸타라,0.872,0.897,0.092,-0.004,720,27,-0.024,0.024,-1.012


In [5]:
# 지속성: 같은 투수가 해마다 낮은가
j = PP[fc.VAL_YEAR][["투수", "gap", "z"]].join(PP[fc.TEST_YEAR][["gap", "z"]], lsuffix=" 2019", rsuffix=" 2020", how="inner")
print(f"투수별 gap 상관 2019 vs 2020: {j['gap 2019'].corr(j['gap 2020']):.3f} (투수 {len(j)}명)")
display(j.sort_values("gap 2019").head(8).round(3))

투수별 gap 상관 2019 vs 2020: -0.147 (투수 37명)


,투수,gap 2019,z 2019,gap 2020,z 2020
pitcher_code,,,,,
65769,김범수,-0.056,-2.214,0.013,0.542
68043,김민,-0.054,-1.711,-0.002,-0.066
64021,박세웅,-0.049,-1.794,0.003,0.134
65933,구창모,-0.045,-2.017,0.013,0.659
69446,원태인,-0.041,-1.614,-0.005,-0.417
67266,박치국,-0.038,-1.468,-0.003,-0.148
66145,김대현,-0.026,-0.811,0.035,2.532
68341,안우진,-0.025,-1.246,0.040,1.443


#### 0-1-3. 상황별 (모든 투수)

- 같은 상황 칸 안 평균 -> gap·llx·rFAST가 0에서 멀면 모델이 그 상황을 모르는 것
- rFAST 표준오차는 공 단위 근사, 오른쪽 두 열은 2020 (같은 방향인지 확인)

In [6]:
def cut(s, b, lab=None):
    return pd.cut(s, b, labels=lab, right=False)

def ctx_cols(d):
    return {"선발/불펜": np.where(d["starter"], "선발", "불펜"),
            "앞선 투구 수": cut(d["pc"], [0, 1, 16, 31, 61, 91, 999], ["0 (첫 타자)", "1–15", "16–30", "31–60", "61–90", "91+"]),
            "같은 타자 몇 번째": d["tto"].clip(upper=3).map({1: "1", 2: "2", 3: "3+"}),
            "이닝": d["inning"].clip(upper=9),
            "아웃": d["out_cn"].clip(upper=2),
            "득점권": np.where(d["base2"] + d["base3"] > 0, "득점권", "아님"),
            "주자": np.where(d["base1"] + d["base2"] + d["base3"] > 0, "있음", "없음"),
            "점수차 (투수 팀)": cut(d["sdiff"], [-99, -3, 0, 1, 4, 99], ["−3↓", "−2~−1", "0", "+1~+3", "+4↑"]),
            "타순": d["batorder"],
            "같은 손": d["same_side"].map({0: "반대 손", 1: "같은 손"}),
            "등판 간격 (일)": cut(d["rest"], [0, 2, 4, 6, 999], ["1", "2–3", "4–5", "6+"]),
            "전날까지 초구 수": cut(d["n_prior"], [0, 1, 51, 201, 501, 1e9], ["0", "1–50", "51–200", "201–500", "501+"]),
            "월": d["month"]}

def ctx_table(d):
    rows = []
    for var, c in ctx_cols(d).items():
        key = pd.Series(np.asarray(c), index=d.index, name="값")
        t = d.groupby(key, observed=True).agg(n=("gap", "size"), 포함률=("hit", "mean"), gap=("gap", "mean"),
                                              llx=("llx", "mean"), rFAST=("rFAST", "mean"), sdF=("rFAST", "std"))
        t["rFAST se"] = t.pop("sdF") / np.sqrt(t.n)
        t.index = pd.MultiIndex.from_product([[var], t.index.astype(str)], names=["변수", "값"])
        rows.append(t)
    return pd.concat(rows)

CT = {yr: ctx_table(D[yr][0]) for yr in D}
both = CT[fc.VAL_YEAR].join(CT[fc.TEST_YEAR][["n", "gap", "rFAST"]], rsuffix=" 2020")
pd.set_option("display.max_rows", 200)
display(both.round(3))

n    포함률    gap    llx  rFAST  rFAST se   n 2020  gap 2020  rFAST 2020
변수         값                                                                                   
선발/불펜      불펜         4242  0.918  0.008 -0.023  0.004     0.007   3979.0     0.004       0.018
           선발         9216  0.879 -0.015  0.044 -0.006     0.005   8636.0    -0.001      -0.001
앞선 투구 수    0 (첫 타자)   1288  0.964  0.056 -0.213  0.185     0.012   1278.0     0.050       0.201
           16–30      2243  0.889 -0.008  0.019 -0.006     0.010   2012.0    -0.004      -0.038
           1–15       3868  0.923  0.018 -0.039  0.012     0.008   3718.0     0.003       0.023
           31–60      3220  0.851 -0.042  0.123 -0.052     0.008   2911.0    -0.015      -0.035
           61–90      2492  0.863 -0.031  0.100 -0.050     0.009   2362.0    -0.004      -0.034
           91+         347  0.862 -0.038  0.119 -0.035     0.026    334.0    -0.031      -0.054
같은 타자 몇 번째 1          7645  0.920  0.017 -0.055  0.039     0.005   7390.0     0.012       0.040
           2          3534  0.854 -0.040  0.119 -0.055     0.008   3276.0    -0.014      -0.044
           3+         2279  0.853 -0.040  0.134 -0.063     0.010   1949.0    -0.020      -0.046
이닝         1          1768  0.924  0.033 -0.103  0.098     0.011   1626.0     0.022       0.111
           2          1680  0.891 -0.004  0.005  0.006     0.012   1596.0     0.010       0.001
           3          1710  0.872 -0.020  0.047 -0.011     0.012   1618.0    -0.021      -0.032
           4          1666  0.864 -0.032  0.106 -0.045     0.012   1621.0    -0.014      -0.036
           5          1616  0.853 -0.043  0.109 -0.045     0.012   1535.0    -0.011      -0.027
           6          1517  0.898 -0.001  0.038 -0.028     0.012   1308.0     0.004      -0.028
           7          1289  0.891 -0.016  0.055 -0.043     0.014   1133.0    -0.004       0.007
           8          1226  0.923  0.012 -0.033  0.014     0.014    990.0     0.010       0.010
           9           986  0.924  0.010 -0.044  0.027     0.015   1188.0     0.014       0.044
아웃         0          4645  0.912  0.014 -0.066  0.067     0.007   4375.0     0.029       0.073
           1          4500  0.886 -0.013  0.052 -0.027     0.007   4222.0    -0.009      -0.019
           2          4313  0.874 -0.026  0.088 -0.052     0.007   4018.0    -0.021      -0.043
득점권        득점권        3346  0.868 -0.030  0.103 -0.079     0.008   3136.0    -0.021      -0.072
           아님        10112  0.899 -0.000 -0.004  0.023     0.005   9479.0     0.008       0.031
주자         없음         7226  0.896 -0.003 -0.003  0.038     0.006   6756.0     0.008       0.048
           있음         6232  0.885 -0.013  0.053 -0.049     0.006   5859.0    -0.008      -0.045
점수차 (투수 팀) +1~+3      3814  0.896 -0.007  0.016 -0.018     0.008   3558.0    -0.006       0.004
           +4↑        1428  0.889 -0.015  0.011  0.039     0.012   1904.0     0.020       0.039
           0          3715  0.900  0.004 -0.006  0.026     0.008   3213.0     0.005       0.034
           −2~−1      3521  0.881 -0.015  0.052 -0.027     0.008   3109.0    -0.011      -0.047
           −3↓         980  0.882 -0.018  0.068 -0.023     0.016    831.0     0.008       0.016
타순         1          1643  0.899  0.006 -0.035  0.047     0.012   1550.0     0.022       0.077
           2          1597  0.908  0.012 -0.018  0.013     0.012   1505.0    -0.024       0.013
           3          1550  0.869 -0.028  0.098 -0.029     0.012   1500.0    -0.015      -0.025
           4          1526  0.874 -0.027  0.075 -0.039     0.012   1472.0    -0.026      -0.031
           5          1513  0.884 -0.016  0.050 -0.039     0.012   1399.0    -0.005       0.002
           6          1480  0.886 -0.016  0.041 -0.019     0.012   1364.0     0.004      -0.014
           7          1439  0.886 -0.015  0.029 -0.013     0.013   1320.0     0.005      -0.030
           8          1396  0.905  0.006 -0.005  0.009     0.013   1282.0     0.029      

#### 0-1-4. 투수별 차이를 상황이 얼마나 설명하나

- 상황 칸 = 앞선 투구 수(0 / 1–15 / 16–30 / 31+) × 재대결 여부 × 아웃 × 득점권 (48칸)
- 상황몫 = 그 투수 공들의 상황 칸 평균 gap (모든 투수로 계산) -> 상황 구성만으로 기대되는 gap
- 남은몫 = gap − 상황몫
- 같은 해 안에서 계산한 진단 (모델 아님)

In [7]:
def decompose(d):
    d = d.copy()
    d["cell"] = (cut(d["pc"], [0, 1, 16, 31, 999]).astype(str) + "|" + (d["tto"] >= 2).astype(str) + "|"
                 + d["out_cn"].clip(upper=2).astype(str) + "|" + (d["base2"] + d["base3"] > 0).astype(str))
    d["gap_ctx"] = d.groupby("cell")["gap"].transform("mean")
    d["gap_res"] = d["gap"] - d["gap_ctx"]
    p = d.groupby("pitcher_code").agg(n=("gap", "size"), gap=("gap", "mean"), 상황몫=("gap_ctx", "mean"), 남은몫=("gap_res", "mean"),
                                      선발비율=("starter", "mean"), 첫타자비율=("pc", lambda s: (s == 0).mean()),
                                      재대결비율=("tto", lambda s: (s >= 2).mean()))
    p.insert(0, "투수", names.reindex(p.index).to_numpy())
    return d, p.sort_values("gap")

DEC = {}
for yr in D:
    d2, p = decompose(D[yr][0])
    DEC[yr] = (d2, p)
    print(f"{yr}: 투수별 gap 분산 {p.gap.var():.5f}, 상황몫 {p['상황몫'].var():.5f}, 남은몫 {p['남은몫'].var():.5f}, "
          f"corr(gap, 상황몫) {p.gap.corr(p['상황몫']):.2f}")
print("\n2019 gap 낮은 10명 / 높은 5명")
p = DEC[fc.VAL_YEAR][1]
display(pd.concat([p.head(10), p.tail(5)]).round(3))

2019: 투수별 gap 분산 0.00086, 상황몫 0.00029, 남은몫 0.00069, corr(gap, 상황몫) 0.46
2020: 투수별 gap 분산 0.00029, 상황몫 0.00005, 남은몫 0.00032, corr(gap, 상황몫) 0.08

2019 gap 낮은 10명 / 높은 5명


,투수,n,gap,상황몫,남은몫,선발비율,첫타자비율,재대결비율
pitcher_code,,,,,,,,
65769,김범수,399,-0.056,-0.007,-0.049,0.689,0.100,0.416
68043,김민,657,-0.054,-0.020,-0.034,0.974,0.041,0.619
64021,박세웅,239,-0.049,-0.019,-0.031,1.000,0.046,0.582
65933,구창모,403,-0.045,-0.014,-0.031,0.940,0.055,0.548
69446,원태인,467,-0.041,-0.016,-0.025,0.940,0.054,0.555
67266,박치국,176,-0.038,0.022,-0.061,0.000,0.267,0.000
66145,김대현,252,-0.026,0.007,-0.033,0.187,0.139,0.187
68341,안우진,371,-0.025,-0.021,-0.004,0.960,0.049,0.617
69045,알칸타라,720,-0.024,-0.020,-0.005,1.000,0.038,0.646


#### 0-1-5. 경기 안 몰림

- 투수×경기마다 놓친 공 수의 (실제 − 기대)² 합 / 독립 가정 분산 합 -> 1이면 공마다 독립, 크면 놓침이 특정 경기에 몰림
- 상황 보정 후 = 기대에 상황몫(0-1-4)을 더한 뒤 다시 계산

In [8]:
def overdisp(d, adj=None):
    pm = (1 - d["exp"] - (0 if adj is None else d[adj])).clip(0, 1)  # 놓칠 확률
    a = d.assign(miss=1 - d["hit"], pm=pm, v=pm * (1 - pm)).groupby(["gameID", "pitcher_code"]).agg(o=("miss", "sum"), e=("pm", "sum"), v=("v", "sum"))
    return ((a.o - a.e) ** 2).sum() / a.v.sum()

display(pd.DataFrame({yr: {"분산비 (상황 보정 전)": overdisp(DEC[yr][0]), "분산비 (상황 보정 후)": overdisp(DEC[yr][0], "gap_ctx")}
                      for yr in D}).round(3))

,2019,2020
분산비 (상황 보정 전),2.306,1.630
분산비 (상황 보정 후),2.103,1.632


In [9]:
# 경기마다 직구 비율이 예측과 얼마나 다른가 (선발, 초구 10개 이상 경기, 2019)
d, P, _ = D[fc.VAL_YEAR]
ds = d[d["starter"]].assign(pF=P[d["starter"].to_numpy(), fc.TYPE_IDX["FAST"]])
gm = ds.groupby(["pitcher_code", "gameID"]).agg(n=("rFAST", "size"), rF=("rFAST", "mean"), pF=("pF", "mean"),
                                                 v=("pF", lambda s: (s * (1 - s)).sum()))
gm = gm[gm.n >= 10]
sd_obs = gm.rF.std()
sd_ind = np.sqrt((gm.v / gm.n ** 2).mean())  # 경기 안 공이 독립이면 기대되는 흩어짐
print(f"선발 경기 {len(gm)}개: 경기별 직구 (실제 − 예측) 표준편차 {sd_obs:.3f}, 독립이면 {sd_ind:.3f} "
      f"-> 경기마다 배합이 달라지는 몫 약 {np.sqrt(max(sd_obs ** 2 - sd_ind ** 2, 0)):.3f}")

선발 경기 386개: 경기별 직구 (실제 − 예측) 표준편차 0.116, 독립이면 0.098 -> 경기마다 배합이 달라지는 몫 약 0.061


#### 0-1-6. 개별 투수 (2019 gap 낮은 6명)

- 구종별 실제 비율 vs 예측 평균, 월별·선발/불펜별 gap·rFAST

In [10]:
d, P, y = D[fc.VAL_YEAR]
for pc_ in PP[fc.VAL_YEAR].index[:6]:
    k = d["pitcher_code"].eq(pc_).to_numpy()
    t = pd.DataFrame({"실제": np.bincount(y[k], minlength=len(fc.CLASSES)) / k.sum(), "예측": P[k].mean(0)}, index=fc.CLASSES)
    t = t[(t.실제 > 0.01) | (t.예측 > 0.01)]
    t["차"] = t.실제 - t.예측
    dk = d[k].assign(역할=np.where(d.loc[k, "starter"], "선발", "불펜"))
    print(f"== {names[pc_]}  초구 {k.sum()}, gap {dk.gap.mean():+.3f}, 시즌 첫 경기 때 전날까지 초구 수 {dk.n_prior.min():.0f}")
    display(t.T.round(3))
    display(pd.concat([dk.groupby("month")[["gap", "rFAST"]].mean().T, dk.groupby("역할")[["gap", "rFAST"]].mean().T], axis=1).round(2))

== 김범수  초구 399, gap -0.056, 시즌 첫 경기 때 전날까지 초구 수 302


,CHUP,CURV,FAST,FORK,SLID
실제,0.110,0.140,0.486,0.055,0.208
예측,0.096,0.138,0.506,0.060,0.199
차,0.014,0.003,-0.020,-0.005,0.009


,3,4,5,6,7,8,9,불펜,선발
gap,0.15,0.12,-0.05,-0.23,-0.03,-0.02,0.06,0.05,-0.10
rFAST,-0.09,-0.08,0.07,-0.16,0.02,-0.10,0.06,-0.02,-0.02


== 김민  초구 657, gap -0.054, 시즌 첫 경기 때 전날까지 초구 수 178


,CHUP,FAST,SLID,TWOS
실제,0.154,0.521,0.169,0.155
예측,0.124,0.555,0.175,0.136
차,0.030,-0.034,-0.006,0.019


,3,4,5,6,7,8,9,불펜,선발
gap,-0.23,0.04,-0.05,-0.23,0.04,-0.02,-0.02,0.15,-0.06
rFAST,-0.38,0.07,-0.06,-0.20,0.06,0.14,-0.11,0.40,-0.05


... (8 outputs left out)

,3,4,5,6,7,8,9,불펜,선발
gap,-0.12,-0.07,-0.01,-0.11,-0.02,-0.06,0.12,-0.15,-0.03
rFAST,-0.02,0.05,0.00,0.00,-0.06,0.12,0.05,0.08,0.01


== 박치국  초구 176, gap -0.038, 시즌 첫 경기 때 전날까지 초구 수 377


,CHUP,CURV,FAST,SLID,TWOS
실제,0.017,0.284,0.648,0.028,0.023
예측,0.032,0.243,0.685,0.023,0.016
차,-0.015,0.041,-0.038,0.005,0.006


,3,4,5,6,7,8,9,불펜
gap,-0.13,-0.15,0.02,-0.02,0.03,-0.09,0.05,-0.04
rFAST,-0.07,-0.07,-0.08,-0.18,0.05,-0.03,0.26,-0.04


### 🔍 0-1 결과
#### 포함률 낮은 투수는 해마다 바뀜 -> 투수 문제가 아니라 상황(첫 타자·재대결·아웃·득점권·타순)을 모르는 문제

- 전체: 포함률 2019 0.891 / 2020 0.901, 기대 0.899 / 0.900 -> 전체 보정은 맞음
- 포함률 낮은 투수 (2019, gap −0.04 ~ −0.06): 김범수·김민·박세웅·구창모·원태인·박치국
  - 경기 단위 |z| > 2는 김범수·구창모뿐, 2020에는 6명 모두 gap ≈ 0 이상 (투수별 gap 상관 2019↔2020 −0.15)
  - -> 투수 고유 특성이 아님, 투수별 보정 하지 않음
- 원인 1: 상황을 모름 (2019·2020 같은 방향, 가장 큼)
  - 그 경기 첫 타자(앞선 투구 수 0): 직구가 예측보다 +0.19 / +0.20, 포함률 0.96 (과소 확신)
  - 앞선 투구 수 31+, 같은 타자 2번째 이상, 2아웃, 득점권, 타순 3–5번, 4–5회: 직구 −0.04 ~ −0.08, gap −0.02 ~ −0.04 (과신)
  - 선발은 이런 상황이 많아 포함률이 낮고(0.879), 불펜은 첫 타자가 많아 높음(0.918)
  - -> 2019 투수별 gap 차이의 약 1/3 (상관 0.46), 2020은 투수별 차이 자체가 잡음 수준
- 원인 2: 역할·시즌 초 배합 변화 (개별 투수)
  - 김범수: 불펜 gap +0.05 / 선발 −0.10 (2019 중 선발 전환), 6월 −0.23
  - 김민: 3월 gap −0.23 (직구 −0.38, 시즌 초 배합 변화를 지난 기록이 늦게 따라감)
  - 원태인: 2019 신인 (시즌 시작 때 기록 0구), 전날까지 초구 1–50개 구간 gap −0.10
- 원인 3: 경기마다 배합이 달라짐 (상황으로 설명 안 됨)
  - 놓친 공 수 분산비 2.3 / 1.6, 상황 보정 후에도 2.1 / 1.6 -> 놓침이 특정 경기에 몰림
  - 선발 경기별 직구 비율이 예측과 다른 몫 SD 약 0.06
  - 그 경기 트래킹 구종은 쓸 수 없음 -> 경기 기록(앞 타석 볼/스트라이크 흐름)·포수·상대 팀으로 일부만 대응 가능
- 반대쪽 (포함률 높음): 함덕주·조상우·주권·박진형 등 구종이 2개 위주인 불펜 -> 과소 확신 (llx −0.10 안팎), 일부는 첫 타자 상황몫
- 영향 작음: 점수 차, 등판 간격, 월 (2019·2020 방향 일관성 약함), 같은 손은 이미 모델에 있음
- 2단계 변수 순서에 반영: ① 앞선 투구 수(첫 타자 여부 포함)·같은 타자 몇 번째 ② 아웃·주자(득점권) ③ 타순 -> 그다음 선발/불펜·기록 적은 투수 처리

### 0-2. 넣을 변수 사전 확인: 묶음 순서 정하기

- 목적: 2단계 구종 모델에 넣을 변수들이 지금 구종 확률(model_03 = model_01, B4)에 얼마나 더해 줄 수 있는지 먼저 재고 넣는 순서를 정함
- 2020은 쓰지 않음 (학습 2017–18, 확인 2019)
- 두 가지로 잼
  1. 변수 하나씩 (모델 없음): 학습 해의 예측 잔차로 "그 변수 칸마다 구종별 배수"를 구해 2019 확률에 곱함 -> log loss 이득
     - 하나씩이라 순서와 무관, 대신 변수끼리 겹치는 몫을 따로 셈
  2. 묶음 단위 (LightGBM): 지금 구종 확률의 log를 출발점(init_score)으로 두고 잔차만 학습 -> 묶음 단독 / A 위에 더하기 / 전부에서 하나씩 빼기
     - 빼 보기 = 순서와 무관한 그 묶음 고유 몫
- 사용하는 정보는 README 3장 "평가 설계" 기준: 경기 기록은 그 타석 앞까지, 트래킹(구종 등)은 전날까지
- 이득 단위: log loss × 1000 (양수 = 좋아짐), 95% 구간은 경기 단위 부트스트랩

In [ ]:
import os, sys, warnings
if not os.path.exists("fp_common.py"):  
    os.chdir("..")
sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore", category=RuntimeWarning)
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.special import log_softmax
import fp_common as fc

m, names = fc.load_data()
fp, y_type, y_zone, val, test = fc.first_pitches(m)
CL, K = fc.CLASSES, len(fc.CLASSES)
TRAIN_YEARS = (2017, 2018)  # 학습 해, 확인은 fc.VAL_YEAR (2019)
pd.set_option("display.max_rows", 200)

#### 0-2-1. 구종 확률 (B4, 2017–2019)

- model_01/08과 같은 함수·파라미터, 2017–18도 같은 방식으로 계산 (학습 데이터 출발점)
- 2017은 지난 시즌 리그 비율이 없음 -> 그 시즌 비율로 대신 (학습에만 쓰는 해라 평가에 새는 것 없음)
  - 처음엔 이걸 빼서 2017 확률이 NaN -> 2017–18 학습 결과가 크게 망가졌음 (2단계 모델에서도 같은 처리 필요)

In [12]:
TYPE_HL, TYPE_ALPHA, TYPE_HAND_K = 25.0, 5.48, 5.48  # model_01

def league_type(year):
    if year == fp["GYEAR"].min():  # 첫 시즌: 지난 기록 없음 -> 그 시즌 비율
        return fp.loc[fp["GYEAR"] == year, CL].mean().to_numpy()
    return fp.loc[fp["GYEAR"] < year, CL].mean().to_numpy()

def type_online(q, year):
    S, N = fc.lookup(fc.prior_table(fp, ["pitcher_code"], CL, TYPE_HL), q, ["pitcher_code"], CL)
    P = fc.shrink(S, N[:, :1], league_type(year), TYPE_ALPHA)
    keys = ["pitcher_code", "same_side"]
    S2, N2 = fc.lookup(fc.prior_table(fp, keys, CL, TYPE_HL), q, keys, CL)
    return fc.shrink(S2, N2[:, :1], P, TYPE_HAND_K)

P = np.full((len(fp), K), np.nan)
for yr in range(TRAIN_YEARS[0], fc.VAL_YEAR + 1):
    k = fp["GYEAR"].eq(yr).to_numpy()
    P[k] = type_online(fp[k], yr)
pp = fc.load_results(["model_03"])["model_03"]
chk = fp[fc.KEYS].assign(_r=np.arange(len(fp))).merge(pp[fc.KEYS + fc.P_TYPE], on=fc.KEYS)
chk = chk[fp["GYEAR"].to_numpy()[chk["_r"]] == fc.VAL_YEAR]
assert np.allclose(P[chk["_r"]], chk[fc.P_TYPE].to_numpy(), atol=1e-9), "model_03 구종 확률 재현 실패"
print("2019 구종 확률 = model_03 (재현 확인)")
yv = fp["pitch_type"].map(fc.TYPE_IDX).to_numpy()
itr = np.flatnonzero(fp["GYEAR"].between(*TRAIN_YEARS).to_numpy())
i19 = np.flatnonzero(fp["GYEAR"].eq(fc.VAL_YEAR).to_numpy())
LL0 = -np.log(np.clip(P[i19, yv[i19]], 1e-12, None))
print(f"학습 {len(itr):,} / 2019 {len(i19):,}, 2019 구종 log loss {LL0.mean():.4f}")

2019 구종 확률 = model_03 (재현 확인)
학습 24,636 / 2019 13,458, 2019 구종 log loss 1.0982


#### 0-2-2. 후보 변수

| 묶음 | 변수 | 정보 종류 |
| --- | --- | --- |
| A 이번 타석 상황 | 앞선 투구 수, 같은 타자 몇 번째, 선발/불펜, 이닝, 아웃, 주자(1·2·3루), 타순, 점수 차 | 경기 기록 |
| B 오늘 경기 흐름 | 오늘 앞 공 볼 비율, 앞 타석 초구 스트라이크 비율, 앞 타석 초구 결과, 앞 타석 결과 | 경기 기록 (같은 경기·같은 투수) |
| C 투수 이력 | 등판 간격, 역할 전환(직전 등판과 선발/불펜 다름), 모든 카운트 직구 비율 / 초구 직구 예측, 직전 경기 직구 비율 − 예측 | 트래킹 전날까지 |
| D 상대 | 홈/원정, 타자 초구 스윙률, 타자·상대 팀 잔차 | 전날까지 |
| E 기타 | 월, 포수·구장 잔차 | 전날까지 |

- 잔차 = 그 타자(포수·팀·구장)를 만났을 때 지금 모델이 구종별로 빗나간 정도 (전날까지 (실제 − 예측) 합 / (공 수 + 50))
- 앞 타석 = 같은 경기에서 이 투수가 상대한 바로 앞 타석 (없으면 "없음" = 그 경기 첫 타자)
- 주자 신원(주자의 타순)은 만들기 비싸서 이번엔 제외 -> A의 주자 결과 보고 결정

In [13]:
m = m.sort_values(fc.KEYS).reset_index(drop=True)
g = m.groupby(["gameID", "pitcher_code"], sort=False)
m["pc"] = g.cumcount()  # 그 경기 앞선 투구 수
m["starter"] = g["inning"].transform("first").eq(1).astype(int)
m["tto"] = m.groupby(["gameID", "pitcher_code", "batter_code"])["pa_id"].transform(lambda s: s.ne(s.shift()).cumsum())
m["sdiff"] = np.where(m["tb"].eq("T"), m["home_score"] - m["away_score"], m["away_score"] - m["home_score"])  # 투수 팀 기준
m["home"] = m["tb"].eq("T").astype(int)  # 초 수비 = 홈 팀 투수
m["isB"] = m["pit_result"].eq("B").astype(float)
m["ball_rate"] = (g["isB"].cumsum() - m["isB"]) / m["pc"].replace(0, np.nan)  # 오늘 앞선 공 볼 비율

# 앞 타석 (같은 경기·같은 투수)
pa = (m.groupby(["gameID", "pitcher_code", "pa_id"], sort=False)
      .agg(first_res=("pit_result", "first"), how=("how", "last"), p0=("pit_no", "min")).reset_index().sort_values(["gameID", "p0"]))
HOW = {**{h: "안타" for h in ["H1", "H2", "H3", "HI", "HB"]}, "HR": "홈런", **{h: "볼넷·사구" for h in ["BB", "IB", "HP"]},
       **{h: "삼진" for h in ["KK", "KN", "KW", "KB", "KP"]}}
pa["how_g"] = pa["how"].map(HOW).fillna("아웃·기타")
pa["first_g"] = pa["first_res"].map({"B": "볼", "T": "루킹", "S": "헛스윙", "V": "헛스윙", "F": "파울", "W": "파울", "H": "인플레이"})
gp = pa.groupby(["gameID", "pitcher_code"], sort=False)
pa["prev_first"] = gp["first_g"].shift().fillna("없음")
pa["prev_how"] = gp["how_g"].shift().fillna("없음")
pa["fs_rate"] = gp["first_g"].transform(lambda s: s.ne("볼").astype(float).cumsum().shift()) / gp.cumcount().replace(0, np.nan)
m = m.merge(pa[["gameID", "pitcher_code", "pa_id", "prev_first", "prev_how", "fs_rate"]], on=["gameID", "pitcher_code", "pa_id"], how="left")

# 등판 간격·역할 전환 (이전 날짜 등판 기준, 같은 날 더블헤더 1차전 제외)
gd = (m.groupby(["pitcher_code", "gameID"], sort=False).agg(game_date=("game_date", "first"), starter=("starter", "first"))
      .reset_index().sort_values("game_date"))
prev = gd[["pitcher_code", "game_date", "starter"]].rename(columns={"starter": "prev_starter"}).assign(prev_date=gd["game_date"])
gd = pd.merge_asof(gd, prev, on="game_date", by="pitcher_code", allow_exact_matches=False)
gd["rest"] = (gd["game_date"] - gd["prev_date"]).dt.days
gd["role_change"] = np.where(gd["prev_starter"].isna(), np.nan, (gd["starter"] != gd["prev_starter"]).astype(float))
m = m.merge(gd[["pitcher_code", "gameID", "rest", "role_change"]], on=["pitcher_code", "gameID"], how="left")

COLS = ["pc", "starter", "tto", "sdiff", "home", "ball_rate", "prev_first", "prev_how", "fs_rate", "rest", "role_change"]
F = fp[fc.KEYS].merge(m[fc.KEYS + COLS], on=fc.KEYS, how="left", validate="one_to_one")
F.index = fp.index
fp = pd.concat([fp, F[COLS]], axis=1)
fp["month"] = fp["game_date"].dt.month
fp["risp"] = ((fp["base2"] + fp["base3"]) > 0).astype(int)
fp["runner_sc"] = fp["base1"].astype(str) + fp["base2"].astype(str) + fp["base3"].astype(str)

In [14]:
# 투수 이력 (트래킹, 전날까지): 모든 카운트 배합(반감 200구), 직전 날짜 경기 배합
m_ok = m[~m["label_doubt"]].copy()
m_ok[CL] = pd.get_dummies(m_ok["pitch_type"]).reindex(columns=CL, fill_value=0).astype(float).to_numpy()
S, N = fc.lookup(fc.prior_table(m_ok, ["pitcher_code"], CL, 200.0), fp, ["pitcher_code"], CL)
mix_all = S / np.maximum(N, 1e-9)
lg = (m_ok.groupby(["pitcher_code", "gameID"]).agg(game_date=("game_date", "first"), **{c: (c, "mean") for c in CL})
      .reset_index().sort_values("game_date"))
q = fp[["pitcher_code", "game_date"]].assign(_r=np.arange(len(fp))).sort_values("game_date")
mix_last = pd.merge_asof(q, lg, on="game_date", by="pitcher_code", allow_exact_matches=False).sort_values("_r")[CL].to_numpy()
iF = CL.index("FAST")
fp["allmix_fast_ratio"] = np.log(np.clip(mix_all[:, iF], 1e-3, None) / P[:, iF])  # 모든 카운트 직구 비율 / 초구 직구 예측 (log)
fp["last_fast_diff"] = mix_last[:, iF] - P[:, iF]  # 직전 경기 직구 비율 − 예측

# 타자 초구 스윙률 (경기 기록, 전날까지, 0-0 공 전부, 리그 쪽으로 20구 당김)
m00 = m[m["ball_before"].eq(0) & m["strike_before"].eq(0)].copy()
m00["swing"] = m00["pit_result"].isin(["S", "F", "H", "W", "V"]).astype(float)
S, N = fc.lookup(fc.prior_table(m00, ["batter_code"], ["swing"]), fp, ["batter_code"], ["swing"])
fp["bat_swing"] = (S[:, 0] + 20 * m00["swing"].mean()) / (N[:, 0] + 20)
fp["bat_n"] = N[:, 0]

# 잔차 인코딩 (전날까지): 그룹별 (실제 원-핫 − 예측) 합 / (공 수 + 50)
RCOLS = ["r_" + c for c in CL]
fp[RCOLS] = fp[CL].to_numpy() - P  # 2017–2019 (2020은 P 없음 -> NaN, 쓰지 않음)
ENC_KEYS = ["batter_code", "catcher_code", "bat_team", "VENUE_ID"]
for key in ENC_KEYS:
    S, N = fc.lookup(fc.prior_table(fp, [key], RCOLS), fp, [key], RCOLS)
    fp[[f"enc_{key}_{c}" for c in CL]] = S / (N[:, :1] + 50.0)
print("변수 만들기 끝")

변수 만들기 끝


#### 0-2-3. 변수 하나씩 (모델 없음)

- 칸마다 구종별 배수 w(구종) = (실제 수 + k·평균 예측) / (예측 합 + k·평균 예측), 2019 확률 × w -> 다시 합 1
- 배수는 학습 해(2017–18) 잔차로 구함 -> 2019에 적용 (전날까지 원칙을 해 단위로 지킴), k는 10·30·100·300 중 2019 1등
  - 1등 k가 300(가장 약한 당김)인데도 음수 = 학습 해 패턴이 2019에 반복되지 않음 = 그 변수 신호 없음
- 참고 열 "2019 교차": 2019 안 경기 단위 5겹 교차 (같은 시즌 패턴, 시즌을 넘는지와 별개)
- 연속 변수는 학습 해 5분위로 칸

In [15]:
def cut(s, b):
    return pd.cut(s, b, right=False).astype(str)

def qbin(s, n=5):
    e = np.unique(np.nanquantile(s.iloc[itr], np.linspace(0, 1, n + 1)))
    e[0], e[-1] = -np.inf, np.inf
    return pd.cut(s, e).astype(str)

V = {
    "앞선 투구 수": ("A", cut(fp["pc"], [0, 1, 16, 31, 61, 91, 999])),
    "같은 타자 몇 번째": ("A", fp["tto"].clip(upper=3)),
    "선발/불펜": ("A", fp["starter"]),
    "이닝": ("A", fp["inning"].clip(upper=9)),
    "아웃": ("A", fp["out_cn"].clip(upper=2)),
    "주자 배치 (8)": ("A", fp["runner_sc"]),
    "득점권": ("A", fp["risp"]),
    "타순": ("A", fp["batorder"]),
    "점수 차": ("A", cut(fp["sdiff"], [-99, -3, 0, 1, 4, 99])),
    "오늘 앞 공 볼 비율": ("B", qbin(fp["ball_rate"])),
    "앞 타석 초구 스트라이크 비율": ("B", qbin(fp["fs_rate"])),
    "앞 타석 초구 결과": ("B", fp["prev_first"]),
    "앞 타석 결과": ("B", fp["prev_how"]),
    "등판 간격": ("C", cut(fp["rest"], [0, 2, 4, 6, 999])),
    "역할 전환": ("C", fp["role_change"]),
    "모든 카운트 직구 비율 / 예측": ("C", qbin(fp["allmix_fast_ratio"])),
    "직전 경기 직구 비율 − 예측": ("C", qbin(fp["last_fast_diff"])),
    "홈/원정": ("D", fp["home"]),
    "타자 초구 스윙률": ("D", qbin(fp["bat_swing"])),
    "상대 팀": ("D", fp["bat_team"]),
    "타자": ("D", fp["batter_code"]),
    "포수": ("E", fp["catcher_code"]),
    "구장": ("E", fp["VENUE_ID"]),
    "월": ("E", fp["month"]),
}
Y = np.eye(K)[yv]
GID = fp["gameID"].to_numpy()

def tilt_fit(b, idx, k):
    O = pd.DataFrame(Y[idx]).groupby(b[idx]).sum()
    E = pd.DataFrame(P[idx]).groupby(b[idx]).sum()
    pi = np.maximum(E.to_numpy() / pd.Series(b[idx]).value_counts().reindex(E.index).to_numpy()[:, None], 1e-6)
    return pd.DataFrame((O.to_numpy() + k * pi) / (E.to_numpy() + k * pi), index=E.index)

def tilt_gain(b, fit_idx, idx, k):
    w = tilt_fit(b, fit_idx, k).reindex(b[idx]).fillna(1.0).to_numpy()
    Q = P[idx] * w
    Q /= Q.sum(1, keepdims=True)
    return -np.log(np.clip(P[idx, yv[idx]], 1e-12, None)) + np.log(np.clip(Q[np.arange(len(idx)), yv[idx]], 1e-12, None))

def game_ci(d, idx, B=1000, seed=0):
    s = pd.DataFrame({"g": GID[idx], "d": d}).groupby("g")["d"].agg(["sum", "count"])
    r = np.random.default_rng(seed).integers(0, len(s), (B, len(s)))
    return np.percentile(s["sum"].to_numpy()[r].sum(1) / s["count"].to_numpy()[r].sum(1), [2.5, 97.5])

ug = np.unique(GID[i19])
fold = pd.Series(np.random.default_rng(0).integers(0, 5, len(ug)), index=ug).reindex(GID[i19]).to_numpy()
rows = []
for name, (bund, b) in V.items():
    b = pd.Series(b).astype(str).fillna("빈 값").to_numpy(dtype=object).astype(str)
    k, d = max(((k, tilt_gain(b, itr, i19, k)) for k in (10.0, 30.0, 100.0, 300.0)), key=lambda t: t[1].mean())
    lo, hi = game_ci(d, i19)
    cf = np.empty(len(i19))
    for f in range(5):
        cf[fold == f] = tilt_gain(b, i19[fold != f], i19[fold == f], k)
    rows.append({"묶음": bund, "변수": name, "칸 수": len(np.unique(b[i19])), "k": k,
                 "이득 (학습 해->2019)": d.mean() * 1000, "95% 하한": lo * 1000, "95% 상한": hi * 1000, "2019 교차": cf.mean() * 1000})
ONE = pd.DataFrame(rows).set_index(["묶음", "변수"])
display(ONE.sort_values("이득 (학습 해->2019)", ascending=False).round(2))

칸 수      k  이득 (학습 해->2019)  95% 하한  95% 상한  2019 교차
묶음 변수                                                                     
B  앞 타석 결과              6   10.0             9.83    7.49   12.39     9.07
   앞 타석 초구 스트라이크 비율     6   30.0             8.18    5.63   10.67     7.38
   앞 타석 초구 결과           6  100.0             7.91    5.60   10.15     8.94
   오늘 앞 공 볼 비율          6  100.0             7.86    5.29   10.27     8.46
A  앞선 투구 수              6  300.0             7.27    3.50   10.36     9.98
   아웃                   3   10.0             6.47    4.71    8.34     7.39
   득점권                  2   10.0             6.27    4.46    8.34     7.61
   주자 배치 (8)            8  300.0             6.20    3.92    8.52     8.32
   이닝                   9  300.0             3.87    1.85    5.85     4.77
   같은 타자 몇 번째           3  300.0             3.03   -0.10    5.85     5.72
D  타자                 258  300.0             1.23   -0.99    3.23     3.21
A  타순                   9  300.0             0.73   -0.65    2.15     1.36
C  등판 간격                5  300.0            -0.49   -1.50    0.60    -0.89
A  점수 차                 5  300.0            -0.63   -2.17    0.78    -0.53
C  모든 카운트 직구 비율 / 예측    5  300.0            -0.94   -2.54    0.61     0.50
A  선발/불펜                2  300.0            -1.12   -2.38   -0.01    -0.43
D  타자 초구 스윙률            5  300.0            -1.26   -2.52   -0.01    -0.03
   홈/원정                 2  300.0            -1.29   -2.36   -0.36    -0.30
C  역할 전환                3  300.0            -2.07   -3.71   -0.72     0.13
   직전 경기 직구 비율 − 예측     6  300.0            -4.31   -6.54   -2.22    -0.52
E  월                    8  300.0            -4.63   -8.27   -1.70    -2.12
   포수                  42  300.0            -5.51   -9.88   -1.53     0.89
D  상대 팀                10  300.0           -21.80  -39.74   -8.85     0.07
E  구장                   9  300.0           -26.12  -39.70  -15.38    -3.28

#### 0-2-4. 묶음 단위 (LightGBM)

- 출발점: 지금 구종 확률의 log (init_score) -> 트리는 잔차만 배움
- 학습 2017–18 (경기 20%는 조기 종료용), 확인 2019, 파라미터는 고정 (작은 트리, 강한 규제): 비교용이지 최종 모델 아님
- 조합: 묶음 단독 / A + 묶음 하나 / 전부 / 전부 − 묶음 하나

In [16]:
for c in ["prev_first", "prev_how"]:
    fp[c + "_cat"] = fp[c].astype("category").cat.codes
enc = lambda key: [f"enc_{key}_{c}" for c in CL]
BUND = {
    "A": ["pc", "tto", "starter", "inning", "out_cn", "base1", "base2", "base3", "batorder", "sdiff"],
    "B": ["ball_rate", "fs_rate", "prev_first_cat", "prev_how_cat"],
    "C": ["rest", "role_change", "allmix_fast_ratio", "last_fast_diff"],
    "D": ["home", "bat_swing", "bat_n"] + enc("batter_code") + enc("bat_team"),
    "E": ["month"] + enc("catcher_code") + enc("VENUE_ID"),
}
BNAME = {"A": "A 이번 타석 상황", "B": "B 오늘 경기 흐름", "C": "C 투수 이력", "D": "D 상대", "E": "E 기타"}
CAT = ["prev_first_cat", "prev_how_cat"]
INIT = np.log(np.clip(P, 1e-6, None))
INIT -= INIT.mean(1, keepdims=True)
gtr = np.unique(GID[itr])
es_g = set(np.random.default_rng(1).choice(gtr, len(gtr) // 5, replace=False))
es = np.isin(GID[itr], list(es_g))
tr, va = itr[~es], itr[es]
PARAMS = dict(objective="multiclass", num_class=K, learning_rate=0.03, num_leaves=7, min_data_in_leaf=200, lambda_l2=10.0,
              feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1, verbose=-1, seed=0, deterministic=True,
              force_row_wise=True, num_threads=4)

def fit_gain(cols):  # 2019 공별 log loss 이득 (지금 확률 − 새 확률), 트리 수
    X = fp[cols]
    cat = [c for c in cols if c in CAT]
    dtr = lgb.Dataset(X.iloc[tr], yv[tr], init_score=INIT[tr], categorical_feature=cat, free_raw_data=False)
    dva = lgb.Dataset(X.iloc[va], yv[va], init_score=INIT[va], categorical_feature=cat, reference=dtr)
    bst = lgb.train(PARAMS, dtr, 2000, valid_sets=[dva], callbacks=[lgb.early_stopping(100, verbose=False)])
    raw = bst.predict(X.iloc[i19], raw_score=True, num_iteration=bst.best_iteration) + INIT[i19]
    return LL0 + log_softmax(raw, axis=1)[np.arange(len(i19)), yv[i19]], bst.best_iteration

runs = {b: [b] for b in BUND} | {"A+" + b: ["A", b] for b in "BCDE"} | {"전부": list(BUND)} | {"전부−" + b: [x for x in BUND if x != b] for b in BUND}
G, rows = {}, []
for key, bs in runs.items():
    G[key], it = fit_gain(sum((BUND[b] for b in bs), []))
    lo, hi = game_ci(G[key], i19)
    rows.append({"조합": " + ".join(BNAME[b] for b in bs) if len(bs) < 5 or key == "전부" else key, "키": key, "트리 수": it,
                 "이득": G[key].mean() * 1000, "95% 하한": lo * 1000, "95% 상한": hi * 1000})
BUN = pd.DataFrame(rows).set_index("키")
display(BUN.round(2))

,조합,트리 수,이득,95% 하한,95% 상한
키,,,,,
A,A 이번 타석 상황,115,28.52,25.08,32.11
B,B 오늘 경기 흐름,54,11.13,8.33,13.84
C,C 투수 이력,33,0.62,-1.42,2.57
D,D 상대,27,3.38,2.20,4.61
E,E 기타,5,0.07,-0.32,0.45
A+B,A 이번 타석 상황 + B 오늘 경기 흐름,115,27.88,24.28,31.46
A+C,A 이번 타석 상황 + C 투수 이력,106,27.60,23.17,32.08
A+D,A 이번 타석 상황 + D 상대,99,26.46,22.93,30.09
A+E,A 이번 타석 상황 + E 기타,75,24.79,21.27,28.14


In [17]:
def diff_row(a, b, label):  # 이득 a − 이득 b
    d = G[a] - G[b]
    lo, hi = game_ci(d, i19)
    return {"비교": label, "차이": d.mean() * 1000, "95% 하한": lo * 1000, "95% 상한": hi * 1000}

rows = [diff_row("A+" + b, "A", f"A 위에 {BNAME[b]} 더함") for b in "BCDE"]
rows += [diff_row("전부", "전부−" + b, f"전부에서 {BNAME[b]} 빼면 잃는 몫") for b in BUND]
display(pd.DataFrame(rows).set_index("비교").round(2))

,차이,95% 하한,95% 상한
비교,,,
A 위에 B 오늘 경기 흐름 더함,-0.64,-2.08,0.73
A 위에 C 투수 이력 더함,-0.92,-3.77,1.86
A 위에 D 상대 더함,-2.06,-4.46,0.27
A 위에 E 기타 더함,-3.73,-6.36,-1.34
전부에서 A 이번 타석 상황 빼면 잃는 몫,11.52,9.23,13.80
전부에서 B 오늘 경기 흐름 빼면 잃는 몫,1.37,0.12,2.50
전부에서 C 투수 이력 빼면 잃는 몫,2.65,0.49,4.76
전부에서 D 상대 빼면 잃는 몫,2.99,1.28,4.68
전부에서 E 기타 빼면 잃는 몫,0.52,-1.51,2.58


#### 0-2-5. A 안에서 하나씩 빼 보기

- A 전부 대비, 그 변수(주자는 1·2·3루 함께)를 빼면 잃는 몫 -> 다른 A 변수로 대신할 수 없는 고유 몫

In [18]:
gA = G["A"]
groups = {"앞선 투구 수": ["pc"], "같은 타자 몇 번째": ["tto"], "선발/불펜": ["starter"], "이닝": ["inning"], "아웃": ["out_cn"],
          "주자 (1·2·3루)": ["base1", "base2", "base3"], "타순": ["batorder"], "점수 차": ["sdiff"]}
rows = []
for nm, cs in groups.items():
    d = gA - fit_gain([c for c in BUND["A"] if c not in cs])[0]
    lo, hi = game_ci(d, i19)
    rows.append({"변수": nm, "빼면 잃는 몫": d.mean() * 1000, "95% 하한": lo * 1000, "95% 상한": hi * 1000})
d = gA - fit_gain([c for c in BUND["A"] if c not in ["pc", "tto", "starter", "inning"]])[0]
lo, hi = game_ci(d, i19)
rows.append({"변수": "투구 수·재대결·선발/불펜·이닝 함께", "빼면 잃는 몫": d.mean() * 1000, "95% 하한": lo * 1000, "95% 상한": hi * 1000})
display(pd.DataFrame(rows).set_index("변수").round(2))

,빼면 잃는 몫,95% 하한,95% 상한
변수,,,
앞선 투구 수,0.86,-0.51,2.39
같은 타자 몇 번째,0.40,-0.02,0.82
선발/불펜,0.27,-0.24,0.78
이닝,0.19,-0.37,0.75
아웃,2.40,1.34,3.57
주자 (1·2·3루),5.33,3.62,7.04
타순,0.70,-0.22,1.76
점수 차,1.68,0.60,2.78
투구 수·재대결·선발/불펜·이닝 함께,10.33,7.55,12.97


### 🔍 0-2 결과
#### A 이번 타석 상황만으로 2019 구종 log loss −0.0285, 나머지 묶음은 A 위에 거의 더하지 못함 
#### 순서: A -> 주자 신원 -> C -> D -> B -> E

- 지금 2019 구종 log loss 1.0982 (이득은 log loss × 1000)
- A 이번 타석 상황이 거의 전부: A만 넣어도 −0.0285 (95% −0.0251 ~ −0.0321), 다섯 묶음 전부 −0.0290 -> 1단계 구종 개선(−0.006)의 약 5배
  - A 안 고유 몫: 주자 5.3 > 아웃 2.4 > 점수 차 1.7 > 타순 0.7(불확실)
  - 투구 수·재대결·선발/불펜·이닝은 서로 대신함 (하나씩 빼면 0.2–0.9, 넷 함께 빼면 10.3) -> 한 덩어리로 넣음
- B 오늘 경기 흐름: 하나씩 볼 때는 가장 큼(앞 타석 결과 9.8)이지만 A 위에서는 −0.6 -> 대부분 "앞 타석 없음 = 그 경기 첫 타자"라 A의 투구 수와 겹침
- C 투수 이력·D 상대: 단독·A 위 모두 작음, 전부에서 빼면 2.7·3.0 -> 다른 묶음과 함께일 때만 작게 보임
- E 기타 (포수·구장·월): 효과 없음, A 위에서는 나빠짐 (−3.7)
  - 하나씩: 구장 −26·상대 팀 −22 -> 학습 해의 팀·구장 잔차가 그 팀 투수 쏠림이라 다음 해에 반복 안 됨
- 전부 − A 대비 = +0.4 (불확실) -> B–E를 다 더해도 A 위에 더 얻는 건 지금 방식으로는 거의 없음
- 순서에서 생기는 차이를 직접 확인: B는 먼저 넣으면 11.1, A 다음이면 −0.6
- 2017 처리: 지난 시즌 리그 비율이 없어 그 시즌 비율로 대신해야 함 (안 하면 2017 확률 NaN -> 학습 실패), 2018만 학습하면 D·E 잔차 기록이 짧아 더 나쁘게 나옴
- 한계: LightGBM 파라미터 고정·작은 트리 -> 비교용, 실제 model_05에서 다시 고름

넣는 순서: A -> 주자 신원(A에서 주자 고유 몫이 가장 커서) -> C -> D -> B -> E
- 이후: 주자 신원은 model_05에 남은 몫이 없어 적용 안 함 (model_06 0-1, README 4-9), C 전에 구종 1단계 보완 추가 (README 5장)


## 모델

In [19]:
# 0장 진단이 m·fp·m_ok 등을 바꿨으므로 모델 전에 다시 불러옴 (1장부터는 이 데이터만 씀)
MODEL = "model_05"
fc._TABLES.clear()  # prior_table 캐시는 id(data)가 키 -> 다시 불러온 데이터가 옛 객체 id를 받아 옛 표를 쓰지 않게 비움
m, names = fc.load_data()
fp, y_type, y_zone, val, test = fc.first_pitches(m)
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]
print(f"초구 {len(fp):,}개 (검증 {val.sum():,} / 평가 {test.sum():,})")

초구 50,709개 (검증 13,458 / 평가 12,615)


## 1. 파라미터

- 구종 B4·위치: model_03 최종값
- LightGBM: 2장에서 고름

In [20]:
# ① 구종 출발점 B4 (model_01, 고정)
TYPE_HL, TYPE_ALPHA, TYPE_HAND_K = 25.0, 5.48, 5.48
# ② 위치 (model_03, 고정)
LOC_REF, LOC_REF_HL = "decay", 173.0
LOC_HL, K_TYPE, K_HAND = 35.4, 1000.0, 17.3
ZONE_M = 32
# 구종 보정 학습 해
TRAIN_FIRST = 2017

## 2. ① 구종: B4 + LightGBM (A 이번 타석 상황)

### 2-1. B4 확률 (출발점)

In [21]:
def league_type(year):
    if year == fp["GYEAR"].min():  # 첫 시즌(2017): 지난 기록 없음 -> 그 시즌 비율 (학습 데이터 출발점용)
        return fp.loc[fp["GYEAR"] == year, CLASSES].mean().to_numpy()
    return fp.loc[fp["GYEAR"] < year, CLASSES].mean().to_numpy()

def type_online(q, year, half_life=TYPE_HL, alpha=TYPE_ALPHA, hand_k=TYPE_HAND_K):  # model_01과 같음 (2017 처리만 추가)
    S, N = fc.lookup(fc.prior_table(fp, ["pitcher_code"], CLASSES, half_life), q, ["pitcher_code"], CLASSES)
    P = fc.shrink(S, N[:, :1], league_type(year), alpha)
    keys = ["pitcher_code", "same_side"]
    S2, N2 = fc.lookup(fc.prior_table(fp, keys, CLASSES, half_life), q, keys, CLASSES)
    return fc.shrink(S2, N2[:, :1], P, hand_k)

K = len(CLASSES)
P_B4 = np.full((len(fp), K), np.nan)
for yr in range(TRAIN_FIRST, fc.TEST_YEAR + 1):
    k = fp["GYEAR"].eq(yr).to_numpy()
    P_B4[k] = type_online(fp[k], yr)
m03 = pd.read_parquet("results/model_03/per_pitch.parquet")
r03 = fp[fc.KEYS].assign(_r=np.arange(len(fp))).merge(m03[fc.KEYS + fc.P_TYPE], on=fc.KEYS)
assert np.allclose(P_B4[r03["_r"]], r03[fc.P_TYPE].to_numpy(), atol=1e-9), "B4가 model_03 구종 확률과 다름"
print("2019·2020 B4 = model_03 구종 확률 (재현 확인)")

2019·2020 B4 = model_03 구종 확률 (재현 확인)


### 2-2. A 이번 타석 상황 변수

- `fc.situation_features` (fp_common): 모든 공(구종 의심 공 포함) 기준, 경기 기록만, 그 공 앞 정보
- `pc` 그 경기 그 투수의 앞선 투구 수 (0 = 그 경기 첫 타자) / `tto` 같은 타자 몇 번째 / `starter` 그 경기 첫 공이 1회 / `sdiff` 투수 팀 기준 점수 차
- 미래 정보 검사 `fc.check_no_future`: 공 300개(그 경기 첫 타자 60개 포함)마다 그 공 직전까지 자른 데이터(`fc.as_of`)로 다시 계산 -> 전체 데이터 값과 같아야 함
  - 자른 데이터: 전날까지 전부 / 오늘 다른 경기·같은 경기 앞 공은 경기 기록만 (트래킹 비움) / 그 공은 투구 전 상태만 (결과·트래킹 비움)
  - 검사가 잡는지 확인: 일부러 미래를 쓴 변수(그 경기 총 투구 수 99/100, 그 공 구종 47/100 = 직구일 때만 다름)는 잡힘 (스크래치 확인)

In [22]:
A_COLS = fc.A_COLS
X_all = fc.situation_features(m, fp)
assert X_all.notna().all().all()
display(X_all.describe().T[["mean", "min", "max"]].round(2))
bad = fc.check_no_future(fc.situation_features, m, fp[fp["GYEAR"].ge(TRAIN_FIRST)], n=300)
assert bad.empty, bad

,mean,min,max
pc,33.39,0.0,124.0
tto,1.59,1.0,5.0
starter,0.69,0.0,1.0
inning,4.55,1.0,12.0
out_cn,0.97,0.0,2.0
base1,0.36,0.0,1.0
base2,0.19,0.0,1.0
base3,0.10,0.0,1.0
batorder,4.81,1.0,9.0
sdiff,0.37,-21.0,19.0


미래 정보 검사: 300개 중 다른 공 0개


### 2-3. LightGBM 파라미터 고르기 (2019)

- 학습 2017–18, 2019 log loss로 고름, 트리 수는 2019 조기 종료 (2019는 선택용이라 허용, 2020은 안 봄)
- learning_rate 0.03, feature_fraction 0.9, bagging 0.8 고정 / 격자: num_leaves × min_data_in_leaf × lambda_l2
- 1등이 후보 범위 끝이면 바깥으로 넓힘 (처음 격자에서 lambda_l2 100·min_data_in_leaf 50이 끝 -> 20·300·1000 추가, 1000도 끝이라 3000 추가)
- 선택 규칙: 2019 1등과 0.0002 안이면 덜 극단적인 조합 (잎 적게 -> 트리 적게)

In [23]:
y_all = fp["pitch_type"].map(fc.TYPE_IDX).to_numpy()
INIT = np.log(np.clip(P_B4, 1e-6, None))
INIT -= INIT.mean(1, keepdims=True)  # 출발점 (공마다 상수는 softmax에 영향 없음)
BASE = dict(objective="multiclass", num_class=K, learning_rate=0.03, feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1,
            verbose=-1, seed=0, deterministic=True, force_row_wise=True, num_threads=4)

def fit_type(train_years, lp, rounds=None, val_year=None):  # 학습 해로 LightGBM 학습 (rounds 없으면 val_year로 조기 종료)
    tr = np.flatnonzero(fp["GYEAR"].between(*train_years).to_numpy())
    dtr = lgb.Dataset(X_all.iloc[tr], y_all[tr], init_score=INIT[tr], free_raw_data=False)
    if rounds is None:
        va = np.flatnonzero(fp["GYEAR"].eq(val_year).to_numpy())
        dva = lgb.Dataset(X_all.iloc[va], y_all[va], init_score=INIT[va], reference=dtr)
        return lgb.train({**BASE, **lp}, dtr, 3000, valid_sets=[dva], callbacks=[lgb.early_stopping(100, verbose=False)])
    return lgb.train({**BASE, **lp}, dtr, rounds)

def type_prob(bst, idx, rounds=None):  # B4 출발점 + 트리 보정 -> 구종 확률 (n, 9)
    raw = bst.predict(X_all.iloc[idx], raw_score=True, num_iteration=rounds or bst.best_iteration) + INIT[idx]
    return np.exp(log_softmax(raw, axis=1))

iv = np.flatnonzero(val)
GRID_LGB = {"num_leaves": [3, 7, 15, 31], "min_data_in_leaf": [20, 50, 200, 800], "lambda_l2": [1.0, 10.0, 100.0, 300.0, 1000.0, 3000.0]}  # 처음 [50, 200, 800]·[1, 10, 100]에서 1등이 끝(λ 100·잎당 50)이라 넓힘, λ 1000도 끝이라 3000까지
rows = []
for nl in GRID_LGB["num_leaves"]:
    for md_ in GRID_LGB["min_data_in_leaf"]:
        for l2 in GRID_LGB["lambda_l2"]:
            lp = {"num_leaves": nl, "min_data_in_leaf": md_, "lambda_l2": l2}
            bst = fit_type((TRAIN_FIRST, fc.VAL_YEAR - 1), lp, val_year=fc.VAL_YEAR)
            rows.append({**lp, "트리 수": bst.best_iteration, "logloss": fc.prob_metrics(type_prob(bst, iv), y_type[val])["logloss"]})
res_lgb = pd.DataFrame(rows).sort_values("logloss").reset_index(drop=True)
TOL = 2e-4
ok = res_lgb[res_lgb["logloss"] <= res_lgb["logloss"].iloc[0] + TOL]
pick = ok.sort_values(["num_leaves", "트리 수"]).iloc[0]  # 평평하면 덜 극단적인 값: 잎 적게 -> 트리 적게 (λ를 키울수록 트리가 늘어남)
LGB_P = {"num_leaves": int(pick.num_leaves), "min_data_in_leaf": int(pick.min_data_in_leaf), "lambda_l2": float(pick.lambda_l2)}
LGB_ROUNDS = int(pick["트리 수"])
b4_ll = fc.prob_metrics(P_B4[iv], y_type[val])["logloss"]
print(f"B4 2019 log loss {b4_ll:.4f} / 1등 {res_lgb.logloss.iloc[0]:.4f} / 선택 {pick.logloss:.4f} {LGB_P}, 트리 {LGB_ROUNDS}개")
display(res_lgb.head(10).round(5))
for k_ in GRID_LGB:  # 파라미터별로 가장 좋은 값 (민감도)
    print(k_, res_lgb.groupby(k_)["logloss"].min().round(5).to_dict())

B4 2019 log loss 1.0982 / 1등 1.0671 / 선택 1.0672 {'num_leaves': 7, 'min_data_in_leaf': 20, 'lambda_l2': 1000.0}, 트리 730개


,num_leaves,min_data_in_leaf,lambda_l2,트리 수,logloss
0,15,50,3000.0,1471,1.06709
1,15,20,3000.0,1471,1.06714
2,15,200,3000.0,1471,1.06714
3,7,20,3000.0,1733,1.06715
4,31,200,3000.0,1471,1.06716
5,7,50,3000.0,1733,1.06717
6,15,50,1000.0,574,1.06717
7,15,200,1000.0,574,1.06718
8,7,20,1000.0,730,1.06720
9,7,50,1000.0,730,1.06720


num_leaves {3: 1.06834, 7: 1.06715, 15: 1.06709, 31: 1.06716}
min_data_in_leaf {20: 1.06714, 50: 1.06709, 200: 1.06714, 800: 1.06749}
lambda_l2 {1.0: 1.07074, 10.0: 1.06869, 100.0: 1.06747, 300.0: 1.06725, 1000.0: 1.06717, 3000.0: 1.06709}


### 🥇1등🥇 LightGBM
#### 2019 구종 log loss 1.0982 (B4) -> 1.0672 (−0.031): 잎 7개, 잎당 최소 20구, lambda_l2 1000, 트리 730개
- lambda_l2가 가장 민감 (1 -> 1000: 1.0707 -> 1.0672) -> 강한 규제가 핵심, 보정은 작고 매끄러워야 함
- 범위 끝(λ 100 -> 1000 -> 3000)을 넓혀 확인, 1000 -> 3000 이득 0.0001로 평평 -> 평평한 구간 규칙으로 λ 1000 (3000은 트리 1,471–1,733개)
- 잎 3개만 조금 나쁨, 잎당 최소 공 수는 800만 아니면 같음

### 2-4. 최종 학습과 구종 확률

- 2019 예측: 2017–18 학습 모델 / 2020 예측: 같은 파라미터·트리 수로 2017–19 다시 학습

In [24]:
BST = {fc.VAL_YEAR: fit_type((TRAIN_FIRST, fc.VAL_YEAR - 1), LGB_P, LGB_ROUNDS),
       fc.TEST_YEAR: fit_type((TRAIN_FIRST, fc.TEST_YEAR - 1), LGB_P, LGB_ROUNDS)}
P_TYPE_ALL = np.full((len(fp), K), np.nan)
for yr, bst in BST.items():
    idx = np.flatnonzero(fp["GYEAR"].eq(yr).to_numpy())
    P_TYPE_ALL[idx] = type_prob(bst, idx, LGB_ROUNDS)
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    a, b = fc.prob_metrics(P_B4[msk], y_type[msk]), fc.prob_metrics(P_TYPE_ALL[msk], y_type[msk])
    print(f"{yr}: log loss B4 {a['logloss']:.4f} -> {b['logloss']:.4f} ({b['logloss'] - a['logloss']:+.4f}), "
          f"top1 {a['top1']:.3f} -> {b['top1']:.3f}, top2 {a['top2']:.3f} -> {b['top2']:.3f}")
imp = pd.DataFrame({yr: pd.Series(b.feature_importance("gain"), index=A_COLS) for yr, b in BST.items()})
display((imp / imp.sum()).sort_values(fc.VAL_YEAR, ascending=False).round(3).rename_axis("변수 (gain 비율)"))

2019: log loss B4 1.0982 -> 1.0672 (-0.0310), top1 0.526 -> 0.534, top2 0.816 -> 0.816
2020: log loss B4 1.0956 -> 1.0610 (-0.0346), top1 0.523 -> 0.531, top2 0.816 -> 0.821


,2019,2020
변수 (gain 비율),,
pc,0.290,0.282
out_cn,0.126,0.133
sdiff,0.118,0.111
batorder,0.101,0.090
inning,0.100,0.098
base2,0.076,0.088
base1,0.074,0.068
tto,0.063,0.070
base3,0.043,0.050


### 2-5. 진단: 상황별로 고쳐졌나 (2019)

- 0-1-3과 같은 표: 직구 실제 − 예측 (rFAST), gap = 포함 − 기대 포함률 (누적 80% 보여 주기)
- B4에서 크던 칸(첫 타자 +0.19, 득점권 −0.08 등)이 0 근처로 오면 보정이 그 상황을 배운 것

In [25]:
def ctx_check(P, msk):
    d = fp[msk].assign(**{c: X_all.loc[msk, c] for c in ["pc", "tto", "starter", "sdiff"]})  # X_all은 fp와 같은 순서
    y = y_type[msk]
    s = fc.show_types(P)
    i = np.arange(len(y))
    d["rFAST"] = (y == fc.TYPE_IDX["FAST"]) - P[:, fc.TYPE_IDX["FAST"]]
    d["gap"] = s[i, y] - (P * s).sum(1)
    cols = {"앞선 투구 수": pd.cut(d["pc"], [0, 1, 16, 31, 61, 999], right=False).astype(str),
            "같은 타자 몇 번째": d["tto"].clip(upper=3).astype(str), "아웃": d["out_cn"].clip(upper=2).astype(str),
            "득점권": np.where(d["base2"] + d["base3"] > 0, "득점권", "아님"), "선발/불펜": np.where(d["starter"] == 1, "선발", "불펜")}
    out = []
    for nm, c in cols.items():
        t = d.groupby(np.asarray(c))[["rFAST", "gap"]].mean()
        t.index = pd.MultiIndex.from_product([[nm], t.index], names=["변수", "값"])
        out.append(t)
    return pd.concat(out)

display(pd.concat({"B4": ctx_check(P_B4[val], val), MODEL: ctx_check(P_TYPE_ALL[val], val)}, axis=1).round(3))
for nm, P in [("B4", P_B4), (MODEL, P_TYPE_ALL)]:
    print(nm, "2019 구종 1등 ECE", round(fc.calib_report(P[val], y_type[val], CLASSES)[0].loc["1등 예측", "ECE"], 4))

B4        model_05       
                      rFAST    gap    rFAST    gap
변수         값                                      
앞선 투구 수    [0, 1)     0.185  0.056    0.021  0.030
           [1, 16)    0.012  0.018   -0.000  0.014
           [16, 31)  -0.006 -0.008   -0.011 -0.013
           [31, 61)  -0.052 -0.042   -0.011 -0.024
           [61, 999) -0.048 -0.032    0.001 -0.020
같은 타자 몇 번째 1          0.039  0.017    0.003  0.009
           2         -0.055 -0.040   -0.011 -0.026
           3         -0.063 -0.040   -0.006 -0.022
아웃         0          0.067  0.014    0.013  0.006
           1         -0.027 -0.013   -0.010 -0.010
           2         -0.052 -0.026   -0.011 -0.012
득점권        득점권       -0.079 -0.030   -0.021 -0.017
           아님         0.023 -0.000    0.004 -0.002
선발/불펜      불펜         0.004  0.008   -0.004  0.009
           선발        -0.006 -0.015   -0.001 -0.012

B4 2019 구종 1등 ECE 0.0128
model_05 2019 구종 1등 ECE 0.0161


In [26]:
# 구종 1등 보정 구간표 (2019): B4 vs model_05
for nm, P in [("B4", P_B4), (MODEL, P_TYPE_ALL)]:
    rep, tabs = fc.calib_report(P[val], y_type[val], CLASSES)
    t = tabs["1등 예측"]
    print(f"{nm}: 1등 ECE {rep.loc['1등 예측', 'ECE']:.4f}")
    display(t[t["공수"] >= 30].round(3).T)

B4: 1등 ECE 0.0128


,30%~40%,40%~50%,50%~60%,60%~70%,70%~80%,80%~90%
공수,1482.000,4294.000,4239.000,2110.000,1136.000,163.000
예측평균,0.366,0.454,0.548,0.646,0.741,0.841
실제비율,0.388,0.451,0.547,0.609,0.708,0.816
차이,0.022,-0.003,-0.001,-0.037,-0.034,-0.025


model_05: 1등 ECE 0.0161


,20%~30%,30%~40%,40%~50%,50%~60%,60%~70%,70%~80%,80%~90%,90%~100%
공수,103.000,1833.000,3808.000,3448.000,2285.000,1308.000,551.000,122.000
예측평균,0.287,0.362,0.453,0.546,0.646,0.744,0.847,0.921
실제비율,0.252,0.369,0.439,0.528,0.626,0.737,0.882,0.967
차이,-0.035,0.008,-0.014,-0.019,-0.020,-0.007,0.035,0.047


### 🔍 상황별 직구 치우침 대부분 사라짐, 보정은 구간마다 방향 다름
#### 첫 타자 +0.185 -> +0.021, 같은 타자 2번째 −0.055 -> −0.011, 득점권 −0.079 -> −0.021 (2019, 직구 실제 − 예측)
- 포함률 gap도 줄었지만 같은 타자 2번째 이상(−0.026)·투구 수 31+(−0.024)는 조금 남음
- 구종 1등 ECE 2019 0.013 -> 0.016: 중간 구간(50–70%) 과신, 높은 구간(80%+) 과소 확신
  - 2019로 맞춘 온도 하나(a 0.97)는 2020 log loss −0.0002뿐, 구종별 재보정은 2020 +0.007 나빠짐 -> 지금은 고치지 않음
  - 다음 묶음마다 ECE·구간표 기록, 커지면 강한 규제(λ 1000)가 큰 보정을 깎는지부터 확인

## 3. ② 위치 (model_03과 같음)

In [28]:
# 24칸 적분 함수는 fp_common으로 옮김: fc.zone_from_gauss_cov(g, q, m=32) (계산 설명은 위, 코드 주석은 fp_common.py)
def zone_probs(g, q, cov=True):  # 24칸 확률: cov=True 공분산 반영 (model_03) / False 독립 가정 (model_02까지)
    return fc.zone_from_gauss_cov(g, q, ZONE_M) if cov else fc.zone_from_gauss(g, q)

In [29]:
L_COLS = MOM + ZONES  # 당길 열: 위치 모멘트 5개(x, 높이 비율, 제곱, 곱) + 24칸 원-핫
K1, K2 = ["pitcher_code", "pitch_type"], ["pitcher_code", "pitch_type", "same_side"]  # 투수×구종 / 투수×구종×손
KR = ["pitch_type", "same_side"]  # 기준점 그룹: 리그 구종×타자 손

def loc_league(q, year, ref="all", ref_hl=None):  # 기준점 (n, 29): "all" 지난 시즌 전부 / "last" 직전 시즌만 / "decay" 전날까지 반감기 ref_hl(공 수) 가중
    past = fp[fp["GYEAR"] < year]  # 지난 시즌까지
    overall = past[L_COLS].mean().to_numpy()  # 그 구종×손 기록이 없을 때 쓰는 리그 전체 평균 (지난 시즌까지)
    if ref == "decay":
        S, N = fc.lookup(fc.prior_table(fp, KR, L_COLS, ref_hl), q, KR, L_COLS)  # 구종×손별 전날까지 가중 합계, 개수
        return np.where(N > 0, S / np.maximum(N, 1e-12), overall)  # 가중 평균, 기록 없으면 리그 전체
    if ref == "last":
        past = fp[fp["GYEAR"] == year - 1]  # 직전 시즌만
    lg = past.groupby(KR)[L_COLS].mean()
    pr = lg.reindex(pd.MultiIndex.from_frame(q[KR]))  # 공마다 그 구종×손 행
    return pr.fillna(pd.Series(overall, index=L_COLS)).to_numpy()

def loc_online(q, year, half_life, k_type, k_hand, ref="all", ref_hl=None):  # 위치 모멘트 + 24칸 비율 (n, 29): L2
    P = loc_league(q, year, ref, ref_hl)  # 출발: 기준점
    S1, N1 = fc.lookup(fc.prior_table(fp, K1, L_COLS, half_life), q, K1, L_COLS)
    P = fc.shrink(S1, N1, P, k_type)  # 투수×구종 쪽으로 k_type만큼
    S2, N2 = fc.lookup(fc.prior_table(fp, K2, L_COLS, half_life), q, K2, L_COLS)
    return fc.shrink(S2, N2, P, k_hand)  # 투수×구종×손 쪽으로 k_hand만큼

def loc_given_type(q, year, t, half_life, k_type, k_hand, ref="all", ref_hl=None, cov=True):  # 구종이 t라고 가정했을 때 24칸 확률 (n, 24)과 좌표 정규분포
    qt = fc.as_type(q, t)  # 구종만 t로 바꾼 사본
    g = fc.gauss(loc_online(qt, year, half_life, k_type, k_hand, ref, ref_hl), qt)  # 모멘트 -> 정규분포 (높이는 그 타자 존으로 cm 환산)
    return zone_probs(g, qt, cov), g  # 정규분포를 24칸 위에서 적분 (cov=True 공분산 반영)

## 4. 실전 예측 (예측 구종으로 위치 이어서)

- model_03과 같음, 구종 확률만 2장 결과(`P_TYPE_ALL`)

In [30]:
def predict(q, year, Pt):  # 초구 q의 공별 예측·점수
    n, i = len(q), np.arange(len(q))
    yt, yz = q["pitch_type"].map(fc.TYPE_IDX).to_numpy(), q["zone"].map(fc.ZONE_IDX).to_numpy()  # 실제 구종·칸 번호
    nll = lambda p: -np.log(np.clip(p, 1e-12, None))  # 공별 log loss (prob_metrics와 같은 방식으로 자름)

    # ① 구종 확률 (n, 9): 2장 결과를 받음

    # ② 위치: 구종 9개 각각 가정 -> 24칸 확률, 실제 위치의 로그 밀도, 예측 중심 (좌우, 높이)
    PZ, LD, MX, MZ = [], [], [], []
    for t in CLASSES:
        P, g = loc_given_type(q, year, t, LOC_HL, K_TYPE, K_HAND, LOC_REF, LOC_REF_HL)
        PZ.append(P); LD.append(fc.log_dens(g, q)); MX.append(g[0]); MZ.append(g[1])
    PZ, LD, MX, MZ = np.stack(PZ, 1), np.stack(LD, 1), np.stack(MX, 1), np.stack(MZ, 1)  # (n, 구종, ...)
    joint = Pt[:, :, None] * PZ  # 구종 확률 × 그 구종일 때 칸 확률 = 구종×칸 결합 확률 (n, 9, 24)
    P_zone = joint.sum(1)  # 실전 24칸 확률 = 구종으로 합침
    # 좌표: 구종 확률로 섞은 정규분포에서 실제 위치의 로그 밀도
    ld_mix = np.log(np.einsum("nt,nt->n", Pt, np.exp(LD - LD.max(1, keepdims=True)))) + LD.max(1)  #  공마다 최댓값 빼기 -> exp -> 섞기 -> log (계산 안전)
    dist_true = np.hypot(q["x"] - MX[i, yt], q["CROSS_PLATE_Z_CM"] - MZ[i, yt])  # 실제 구종 중심과의 거리 cm
    dist_mix = np.hypot(q["x"] - (Pt * MX).sum(1), q["CROSS_PLATE_Z_CM"] - (Pt * MZ).sum(1))  # 구종 확률로 가중한 중심과의 거리 cm

    t_hat = np.array(CLASSES)[Pt.argmax(1)]  # 1등 구종 이름

    # 공별 결과 표 (필수: 키, 구종 확률 9개, 실전 24칸 확률 / 나머지는 선택 열)
    return pd.concat([
        q[["gameID", "pit_no", "pitcher_code", "GYEAR", "pitch_type", "zone"]].reset_index(drop=True),
        pd.DataFrame(Pt, columns=fc.P_TYPE),  # p_type_<구종>
        pd.DataFrame(P_zone, columns=fc.P_ZONE),  # p_zone_<칸>
        pd.DataFrame({"t_hat": t_hat}),  # 1등 구종 이름
        pd.DataFrame({
            "ll_type": nll(Pt[i, yt]),  # 구종 log loss
            "ll_zone": nll(P_zone[i, yz]),  # 24칸 log loss (실전)
            "ll_zone_true": nll(PZ[i, yt, yz]),  # 24칸 log loss (실제 구종)
            "ll_joint": nll(joint[i, yt, yz]),  # 구종×칸 log loss
            "nll_coord": -ld_mix,  # 좌표 NLL (실전)
            "nll_coord_true": -LD[i, yt],  # 좌표 NLL (실제 구종)
            "dist": np.asarray(dist_mix),  # 좌표 거리 cm (실전)
            "dist_true": np.asarray(dist_true),  # 좌표 거리 cm (실제 구종)
        })], axis=1)

In [31]:
# 2019 검증 + 2020 평가 초구 전부 예측
pp = pd.concat([predict(q_val, fc.VAL_YEAR, P_TYPE_ALL[val]), predict(q_test, fc.TEST_YEAR, P_TYPE_ALL[test])], ignore_index=True)
print(pp.shape)
# 확인: 구종과 무관한 열(실제 구종 기준 위치)은 model_03과 같아야 함
chk = pp.merge(m03, on=fc.KEYS, suffixes=("", "_03"))
same = ["ll_zone_true", "nll_coord_true", "dist_true"]
bad = [c for c in same if not np.allclose(chk[c], chk[c + "_03"], equal_nan=True)]
assert len(chk) == len(pp) and not bad, f"model_03과 다른 열: {bad}"
print("실제 구종 기준 위치 = model_03 (재현 확인)")

(26073, 48)
실제 구종 기준 위치 = model_03 (재현 확인)


## 5. 평가

- 채점은 `fc.evaluate`, model_03과 나란히 / 경기 단위 부트스트랩 (model_05 − model_03, 2020)
- 2020은 확인용 -> 2020을 보고 파라미터 다시 고르지 않음

In [32]:
def eval_years(pp_):  # 2019·2020 주요 지표 표
    return pd.DataFrame({"검증 2019": fc.evaluate(pp_, fp, fc.VAL_YEAR), "평가 2020": fc.evaluate(pp_, fp, fc.TEST_YEAR)})

summary = eval_years(pp)
m03_sum = eval_years(m03)
display(pd.concat({MODEL: summary, "model_03": m03_sum, "차이 (model_03 대비)": summary - m03_sum}, axis=1).round(4))

model_05          model_03          차이 (model_03 대비)        
                     검증 2019  평가 2020  검증 2019  평가 2020          검증 2019 평가 2020
구종 logloss            1.0672   1.0610   1.0982   1.0956          -0.0310 -0.0346
구종 top1               0.5343   0.5306   0.5258   0.5233           0.0085  0.0072
구종 top2               0.8156   0.8207   0.8157   0.8162          -0.0001  0.0045
구종 top3               0.9357   0.9387   0.9331   0.9359           0.0026  0.0029
구종 macro F1           0.3238   0.3361   0.2476   0.2857           0.0762  0.0504
24칸 logloss (실전)      2.9496   2.9455   2.9512   2.9479          -0.0016 -0.0024
24칸 top1              0.1291   0.1338   0.1286   0.1330           0.0004  0.0008
24칸 top3              0.2902   0.2906   0.2873   0.2897           0.0029  0.0010
H/S/C logloss         1.0748   1.0767   1.0753   1.0772          -0.0005 -0.0005
H/S/C 정확도             0.3952   0.3889   0.3935   0.3885           0.0017  0.0004
24칸 logloss (실제 구종)   2.8933   2.8967   2.8933   2.8967           0.0000  0.0000
구종×칸 logloss          3.9605   3.9569   3.9914   3.9916          -0.0310 -0.0346
좌표 평균거리 cm           32.6913  31.8324  32.7396  31.8837          -0.0483 -0.0512
좌표 NLL                9.3175   9.2660   9.3195   9.2690          -0.0020 -0.0029

In [33]:
display(fc.bootstrap_diff(fc.pitch_scores(m03, fp), fc.pitch_scores(pp, fp)).round(4))  # 음수 = model_05가 좋음
# 실전 출력: 누적 80% 보여 주기 (2020)
for nm, P_ in [("model_03", m03), (MODEL, pp)]:
    d = fc.align(P_, fp, fc.TEST_YEAR)
    Pt_, yt_ = d[fc.P_TYPE].to_numpy(), d["pitch_type"].map(fc.TYPE_IDX).to_numpy()
    s = fc.show_types(Pt_)
    print(f"{nm}: 보여 준 구종 수 평균 {s.sum(1).mean():.2f}, 포함률 {s[np.arange(len(yt_)), yt_].mean():.3f}")

,차이 (b−a),95% 하한,95% 상한,판정
ll_type,-0.0346,-0.0386,-0.0303,좋아짐
ll_zone,-0.0024,-0.0035,-0.0012,좋아짐
ll_joint,-0.0346,-0.0386,-0.0303,좋아짐
nll_coord,-0.0029,-0.0042,-0.0016,좋아짐
dist,-0.0512,-0.0731,-0.0277,좋아짐


model_03: 보여 준 구종 수 평균 2.45, 포함률 0.901
model_05: 보여 준 구종 수 평균 2.39, 포함률 0.902


### 🥇1등🥇
#### 2020 구종 log loss 1.0956 -> 1.0610 (−0.035), 부트스트랩 6개 점수 모두 좋아짐
- 2019 −0.031 -> 2020 −0.035: 2020이 더 큼 -> 2019 과적합 아님
- 1단계 전체 구종 개선(베이스라인 -> model_03 −0.006)의 약 6배
- 24칸 실전 −0.0024, 구종×칸 −0.035, 좌표 거리 −0.05cm
- 위치는 구종을 맞혀도 조금만 좋아짐 -> 위치 구분력 문제는 그대로 (3단계)
- 누적 80% 보여 주기: 평균 2.45개 -> 2.39개, 포함률 90.1% -> 90.2% (적게 보여 주고 같은 포함률)
- 구종 macro F1 0.286 -> 0.336, top1 0.523 -> 0.531

## 6. 저장

- `results/model_05/` -> `compare.ipynb` 실행하면 자동 포함

In [34]:
fc.save_results(MODEL, pp, {"summary": summary, "lgb_grid": res_lgb})  # per_pitch.parquet + summary.csv + LightGBM 격자

저장: results\model_05 ['lgb_grid.csv', 'per_pitch.parquet', 'summary.csv']
